# Dividing Large Annotation File Into Smaller Files

In [ ]:
import json
import os

def split_json_file(input_path, output_dir, chunk_size=5):  # 5 paragraph in each file
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if 'classes' not in data or 'annotations' not in data:
        raise ValueError('Input JSON must contain \'classes\' and \'annotations\'')

    classes = data['classes']
    annotations = data['annotations']

    if not isinstance(annotations, list):
        raise ValueError('\'annotations\' must be a list')

    os.makedirs(output_dir, exist_ok=True)

    total_annotations = len(annotations)
    file_count = 1

    for i in range(0, total_annotations, chunk_size):
        chunk = annotations[i:i + chunk_size]

        output_data = {
            'classes': classes,
            'annotations': chunk
        }

        output_file_name = f'split_{file_count:03d}.json'
        output_path = os.path.join(output_dir, output_file_name)

        with open(output_path, 'w', encoding='utf-8') as out_f:
            json.dump(output_data, out_f, ensure_ascii=False, indent=4)

        file_count += 1

    print(f'Total annotations: {total_annotations}')
    print(f'Created {file_count - 1} split files in {output_dir}')

In [ ]:
input_file = '/home/umayer/Work/research/ner_data/annotation/Koshkava_2014/Koshkava_2014_spacy.json'
output_folder = '/home/umayer/Work/research/ner_data/annotation/Koshkava_2014'

split_json_file(input_file, output_folder)


# Format Conversion

In [1]:
import json
import datetime

## 1. Exclude Period in Annotations

In [2]:
def period_checker(annotation_file):
    with open(annotation_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for ann in data['annotations']:
        for ent in ann[1]['entities']:
            start = ent[0]
            end = ent[1]
            ent_mention = ann[0][start:end]
            if ent_mention[-1] == '.':
                new_end = end - 1
                ent[1] = new_end
                print(f'Initial {end} ; Updated {new_end} ; "{ent_mention}"')

    return data

## 1. Convert Spacy-to-AnNER Format

In [3]:
def convert_to_anner(spacy_data, filename, annotator):
    STATUS = 'Candidate'
    COLOR_LIST = [
        'red-11', 'blue-11', 'light-green-11', 'yellow-11', 
        'purple-11', 'orange-11', 'teal-11', 'pink-11', 
        'brown-11', 'cyan-11', 'lime-11'
    ]
    
#     with open(input_file, 'r', encoding='utf-8') as file:
#         data = json.load(file)

    # Get timestamp
    timestamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    # Build classes dynamically from input
    classes = []
    class_name_to_id = {}
    for idx, class_name in enumerate(spacy_data['classes'], start=1):
        color = COLOR_LIST[(idx - 1) % len(COLOR_LIST)]  # Rotate colors if needed
        classes.append({
            'id': idx,
            'name': class_name,
            'color': color
        })
        class_name_to_id[class_name] = class_name  # Just map to itself for annotation use

    # Build annotations
    annotations = []
    for text, ann in spacy_data['annotations']:
        entities = []
        for start, end, label in ann['entities']:
            entities.append([
                None,
                start,
                end,
                [
                    [
                        STATUS,
                        label,
                        timestamp,
                        annotator
                    ]
                ]
            ])
        annotations.append([None, text, {'entities': entities}])

    # Final structure
    anner_data = {
        'classes': classes,
        'annotations': annotations
    }
    
    # Write to output file
    with open(filename, 'w', encoding='utf-8') as file:
        json.dump(anner_data, file, indent=2)

    print(f'Conversion completed. Output saved to {filename}')


In [4]:
# convert_to_anner(
#     spacy_data=period_checker('/home/umayer/Work/research/ner_data/annotation/Koshkava_2014/Koshkava_2014_8of8_AnNER.json'),
#     filename='Koshkava_2014_8of8_AnNER.json',
#     annotator='gpt-4o'
# )


In [6]:
convert_to_anner(
    spacy_data=period_checker('/home/umayer/Work/research/ner_data/annotation/Wolf_2018/Wolf_2018_spacy.json'),
    filename='/home/umayer/Work/research/ner_data/annotation/Wolf_2018/Wolf_2018_AnNER.json',
    annotator='gpt-4o'
)

Conversion completed. Output saved to /home/umayer/Work/research/ner_data/annotation/Wolf_2018/Wolf_2018_AnNER.json


# Checking Entity Span from Start and End Position

In [ ]:
import json

def print_entities_paragraph_wise(input_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if 'annotations' not in data:
        raise ValueError('Input JSON must contain \'annotations\'')

    annotations = data['annotations']
    if not isinstance(annotations, list):
        raise ValueError('\'annotations\' must be a list')

    for p_idx, ann in enumerate(annotations, start=1):
        if not isinstance(ann, list) or len(ann) < 3:
            continue

        text = ann[1]
        meta = ann[2] if isinstance(ann[2], dict) else {}
        entities = meta.get('entities', [])

        print(f'Paragraph {p_idx}:')
        print('start\tend\tentity')

        if not entities:
            print('(no entities)\n')
            continue

        for ent in entities:
            # Expected structure: [null, start, end, [[..., LABEL, ...], ...]]
            if not isinstance(ent, list) or len(ent) < 3:
                continue

            start = ent[1]
            end = ent[2]

            if not isinstance(start, int) or not isinstance(end, int):
                continue

            if start < 0 or end > len(text) or start >= end:
                continue

            entity_text = text[start:end]
            print(f'{start}\t{end}\t{entity_text}')

        print('')  # blank line between paragraphs


In [ ]:
print_entities_paragraph_wise('/home/umayer/Downloads/TCASE2-annotations.json')
# or your absolute path:
# print_entities_paragraph_wise('/mnt/data/Koshkava_2014_3of8_AnNER.json')
